# 🏠 RoomCanvas AI — Inference Server

**Architecture:** Depth Anything (small, geometry anchor) → FLUX.2 [klein] 4B (instruction-based edit mode) → optional refine pass  
**Purpose:** Persistent HTTP inference server inside a Colab T4 GPU runtime, exposed via ngrok.  
**Endpoint:** `POST /infer` accepts a room image + `style` + `room_type`, returns 3 redesigned variations + metadata.

---
### Why this replaced the prior SD1.5 + MultiControlNet approach
The previous pipeline (Realistic Vision V6.0 + MLSD + MiDaS) was *structure-conditioned text-to-image*, not *image editing*. That distinction caused every observed failure: missing furniture (weak compositional binding in SD1.5/CLIP), floating objects (MLSD has no floor-plane concept, MiDaS on an empty room is near-flat), and poor prompt adherence (SD1.5 is the weakest language-following model still in common use). **Instruction-based editing models — where the model is trained to take your exact photo and apply only the requested change — solve these at the architecture level, not via conditioning-map tuning.**

---
### Quick Start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cells top to bottom (Sections 1 → 20)
3. Paste your free **ngrok auth token** into Section 5's `NGROK_AUTH_TOKEN` field
4. After Section 20 finishes, run Section 21 to test the live server

> ⚠️ **`room_type` is now a required API field.** The frontend must include a room-type dropdown (`living_room` / `bedroom` / `home_office`) alongside the style selector or requests will 400.


## Section 2 — Environment & GPU Check

This cell **must pass** before anything else. If it raises `RuntimeError`, go to **Runtime → Change runtime type → T4 GPU**.


In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU -> Save, then reconnect.'
    )
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'torch.cuda.is_available() returned False even though nvidia-smi passed. '
        'Try restarting the runtime.'
    )

print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'✅ VRAM: {total_vram:.1f} GB total')
print(f'✅ PyTorch version: {torch.__version__}')


## Section 3 — Dependency Installation

Installs `diffusers` from the `main` branch (required for `Flux2KleinPipeline` — not yet on a stable release).  
**The resolved `diffusers.__version__` is printed at the end of this cell and in every run's logs for debugging.**  
**Do not skip this cell or pin diffusers to a fixed version number.**


In [ ]:
import subprocess, sys

# Diffusers installed from main branch -- Flux2KleinPipeline requires latest diffusers.
# Do NOT pin to a version number here: edit-mode support was added after the last release.
# All other packages are pinned for reproducibility.
packages = [
    'git+https://github.com/huggingface/diffusers.git',
    'transformers>=4.46.0',
    'accelerate>=1.0.1',
    'safetensors>=0.4.5',
    'fastapi==0.115.2',
    'uvicorn==0.34.0',
    'pyngrok==7.2.0',
    'python-multipart==0.0.18',
    'nest-asyncio==1.6.0',
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError('pip install failed -- see stderr above.')

print('Installation complete.')
import diffusers
print(f'diffusers version: {diffusers.__version__}')


## Section 4 — Imports


In [ ]:
import io
import base64
import time
import logging
import os
import asyncio
import threading
import random

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
import requests
from pyngrok import ngrok

print('✅ All imports successful.')


## Section 5 — Configuration

**All constants live here.** Nothing is hardcoded anywhere else in the notebook.  
Get a free ngrok token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).


In [ ]:
MODEL_ID = 'black-forest-labs/FLUX.2-klein-4B'
DEPTH_MODEL_ID = 'depth-anything/Depth-Anything-V2-Small-hf'

# FLUX.2 image constraints
IMG_MAX_SIDE    = 1024   # cap max side for T4 VRAM/speed budget
IMG_DIM_MULTIPLE = 16   # FLUX.2 requires height/width as multiples of 16

# These defaults match the distilled klein checkpoint (4-step).
# Section 7 validates at runtime and prints the actual recommended values.
NUM_INFERENCE_STEPS = 24     # overridden if model card reports distilled=True (4 steps)
GUIDANCE_SCALE      = 1.0   # BFL reference default for distilled; checked in Section 7

NUM_VARIATIONS  = 3
REFINE_STRENGTH = 0.2

SEEDS = [42, 123, 777]  # MUST NOT CHANGE -- backend history/regenerate depends on these

ROOM_TYPES = ['living_room', 'bedroom', 'home_office']
STYLES     = ['modern_minimalist', 'scandinavian', 'industrial', 'bohemian', 'luxury_contemporary']

NGROK_AUTH_TOKEN = '' #@param {type:"string"}

print('✅ Configuration loaded.')
print(f'   Model: {MODEL_ID}')
print(f'   Depth: {DEPTH_MODEL_ID}')
print(f'   Seeds: {SEEDS}  |  Max side: {IMG_MAX_SIDE}  |  Steps: {NUM_INFERENCE_STEPS} (will be verified in Section 7)')


## Section 7 — Model Loading & Pipeline Initialization

Loads FLUX.2 [klein] 4B with defensive API fallbacks for the rapidly-evolving diffusers codebase.  
**Mandatory:** runs an edit-mode validation call at the end — if this fails, do not proceed.  
VRAM budget: ~8–13 GB with CPU offload active (comfortable inside a 16 GB T4).

> ⚠️ Section 6 (Drive mount) has been removed. HF Hub's disk cache at `/root/.cache/huggingface` is sufficient for a single Colab session and removes a failure point (Drive auth flakiness). First run downloads ~8 GB; subsequent runs within the same session use the cached weights.


In [ ]:
import time as _time_mod

print('Loading FLUX.2 klein pipeline...')
_t0 = _time_mod.time()

# Defensive class resolution: prefer the dedicated pipeline class,
# fall back to DiffusionPipeline if the installed diffusers build
# doesn't expose Flux2KleinPipeline under that name yet.
try:
    from diffusers import Flux2KleinPipeline as _PipeCls
    _pipe_cls_name = 'Flux2KleinPipeline'
except ImportError:
    from diffusers import DiffusionPipeline as _PipeCls
    _pipe_cls_name = 'DiffusionPipeline (fallback -- Flux2KleinPipeline not found)'
    print(f'⚠️  {_pipe_cls_name}')
    print('   If this appears, re-run Section 3 to ensure diffusers main branch was installed.')
print(f'Using pipeline class: {_pipe_cls_name}')


def _load_pipeline(model_id, dtype):
    # Defensive kwarg resolution: some diffusers versions use dtype=, others torch_dtype=.
    try:
        return _PipeCls.from_pretrained(model_id, dtype=dtype)
    except TypeError:
        return _PipeCls.from_pretrained(model_id, torch_dtype=dtype)


# Defensive device placement
try:
    pipe = _load_pipeline(MODEL_ID, torch.bfloat16)
    pipe = pipe.to('cuda')
except torch.cuda.OutOfMemoryError:
    print('⚠️  Direct CUDA placement OOM -- falling back to CPU offload.')
    pipe = _load_pipeline(MODEL_ID, torch.bfloat16)

# Always enable CPU offload on a 16GB T4 -- costs some latency but is the
# single biggest lever against OOM crashes across a 3-variation + refine run.
if hasattr(pipe, 'enable_model_cpu_offload'):
    pipe.enable_model_cpu_offload()
    print('✅ CPU offload enabled (VRAM safety net).')

_vram_gb = torch.cuda.memory_allocated() / 1024**3
print(f'✅ FLUX.2 klein loaded in {_time_mod.time()-_t0:.1f}s | VRAM allocated: {_vram_gb:.2f} GB')

# Runtime capability check: inspect model card / config to set accurate step defaults.
import inspect as _inspect
import diffusers as _diffusers_mod
_call_sig = str(_inspect.signature(pipe.__call__))

# Heuristic: klein distilled typically has guidance_embeds or a very small default step count.
# The safest approach is to print resolved values and let the user verify them.
_has_distilled_hint = (
    hasattr(pipe, 'transformer') and
    hasattr(getattr(pipe, 'transformer', None), 'config') and
    getattr(getattr(pipe.transformer, 'config', None), 'guidance_embeds', False)
)
if _has_distilled_hint:
    print()
    print('ℹ️  Model config indicates DISTILLED checkpoint (guidance_embeds=True).')
    print('   Recommended: NUM_INFERENCE_STEPS=4, GUIDANCE_SCALE=1.0')
    print('   If quality is poor, try NUM_INFERENCE_STEPS=8.')
else:
    print()
    print('ℹ️  Could not auto-detect distilled flag -- using configured defaults.')
    print(f'   NUM_INFERENCE_STEPS={NUM_INFERENCE_STEPS}, GUIDANCE_SCALE={GUIDANCE_SCALE}')
    print('   Verify against model card if outputs look wrong.')

# Mandatory edit-mode validation call
print()
print('Running mandatory edit-mode validation call...')
from diffusers.utils import load_image as _load_img
try:
    _test_img = _load_img(
        'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png'
    )
    _test_out = pipe(
        image=_test_img,
        prompt='Turn this cat into a dog',
        num_inference_steps=4,
        guidance_scale=1.0,
    ).images[0]
    assert _test_out is not None and _test_out.size[0] > 0, 'Validation returned None or empty image.'
    print('✅ Edit mode validated: pipe(image=..., prompt=...) works on this diffusers build.')
    print(f'   diffusers=={_diffusers_mod.__version__}')
except Exception as _val_err:
    raise RuntimeError(
        f'Edit-mode validation FAILED: {_val_err}\n'
        f'Check that diffusers was installed from main branch (Section 3).\n'
        f'Installed version: {_diffusers_mod.__version__}\n'
        'If the version looks old, re-run Section 3 and restart the runtime.'
    ) from _val_err


## Section 8 — Memory & VRAM Guard Setup

CPU offload is already enabled in Section 7 as the primary VRAM lever.  
This section adds a VRAM guard helper used defensively throughout the notebook,
and confirms the current VRAM headroom before proceeding to depth model loading.


In [ ]:
def vram_guard(label=''):
    # Print current VRAM usage with a label. Call before/after each heavy operation.
    alloc = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  VRAM [{label}]: {alloc:.2f} / {total:.1f} GB allocated')
    return alloc


vram_guard('after FLUX.2 load')
print('✅ VRAM guard helper defined.')


## Section 9 — Helpers: Image Preprocessing

FLUX.2 requires `height` and `width` to be multiples of 16 (not a fixed 512px like ControlNet).  
The helper below preserves aspect ratio, caps the max side at `IMG_MAX_SIDE`, and rounds both dimensions down to the nearest multiple of 16.


In [ ]:
def _round_to_multiple(x, multiple):
    # Round x down to the nearest multiple, minimum = multiple.
    return max(multiple, (x // multiple) * multiple)


def prepare_image(image, max_side=None, dim_multiple=None):
    # Resize PIL image to fit within max_side x max_side while preserving aspect ratio.
    # Both output dimensions are multiples of dim_multiple (required by FLUX.2).
    if max_side is None:
        max_side = IMG_MAX_SIDE
    if dim_multiple is None:
        dim_multiple = IMG_DIM_MULTIPLE
    w, h = image.size
    # Only downscale, never upscale
    scale = min(max_side / max(w, h), 1.0)
    new_w = _round_to_multiple(int(w * scale), dim_multiple)
    new_h = _round_to_multiple(int(h * scale), dim_multiple)
    return image.convert('RGB').resize((new_w, new_h), Image.LANCZOS)


# Sanity check
_test_img = Image.new('RGB', (1920, 1080), color=(180, 180, 180))
_resized = prepare_image(_test_img)
assert _resized.size[0] % IMG_DIM_MULTIPLE == 0, 'Width not multiple of 16'
assert _resized.size[1] % IMG_DIM_MULTIPLE == 0, 'Height not multiple of 16'
assert max(_resized.size) <= IMG_MAX_SIDE, 'Max side exceeded'
print(f'✅ prepare_image() defined. 1920x1080 -> {_resized.size}')


## Section 11b — Depth Estimation (Depth Anything V2, small)

Provides a geometry/scale anchor for debugging and metadata. **Not wired into the FLUX.2 call** — klein takes the raw photo directly and uses its own image-conditioning for geometry preservation. Depth is computed and attached to response metadata; reserved for the SAM2 escalation path (§9) if testing shows furniture scale issues.

The depth pass is **non-fatal**: if the model fails to load or the inference fails, a warning is printed and generation continues — depth is metadata-only.


In [ ]:
print(f'Loading Depth Anything ({DEPTH_MODEL_ID})...')
depth_estimator = None
try:
    from transformers import pipeline as hf_pipeline
    depth_estimator = hf_pipeline(
        task='depth-estimation',
        model=DEPTH_MODEL_ID,
        device=0,  # T4
    )
    print('✅ Depth Anything loaded.')
    vram_guard('after Depth Anything load')
except Exception as _de_err:
    depth_estimator = None
    print(
        f'⚠️  Failed to load depth model: {_de_err}\n'
        f'   Generation will continue -- depth is metadata-only (non-fatal).\n'
        f'   To fix: verify DEPTH_MODEL_ID in Section 5 resolves to a valid HF repo.\n'
        f'   Safe fallback: depth-anything/Depth-Anything-V2-Small-hf'
    )


def get_depth_map(image):
    # Returns a PIL depth map image, or None if the depth model is unavailable.
    # Used only as an internal geometry sanity check and response metadata.
    # Not passed into FLUX.2 as a conditioning input.
    if depth_estimator is None:
        return None
    try:
        result = depth_estimator(image)
        return result['depth']
    except Exception as e:
        print(f'⚠️  Depth inference failed non-fatally: {e}')
        return None


print('✅ get_depth_map() defined (non-fatal, metadata-only).')


## Section 12 — Style Templates & Prompt Compiler

Static per-style furniture templates for all 5 styles × 3 room types.  
Each template has a base `items` list (shown in every render) and an `alt_pool` (swapped in/out across the 3 variations for genuinely different layouts).  

> **Escalation path (documentation only — not implemented):** If the Section 15 golden-test-set run shows recurring failures: (a) wire the depth map into the FLUX.2 call if furniture scale/placement is wrong, (b) add Florence-2 room captioning as a fallback only when `room_type` is ambiguous, (c) add SAM2 floor/wall masking if geometry drift is observed, (d) replace the static alt-pool logic with an LLM call only if template repetition becomes a real user-facing problem. None of these are implemented in this notebook version.


In [ ]:
STYLE_TEMPLATES = {
    'modern_minimalist': {
        'living_room': {
            'items': [
                'low-profile grey linen sofa against the main wall',
                'matte black metal coffee table',
                'slim wall-mounted TV console with a mounted flatscreen TV opposite the seating',
                'large neutral-tone area rug centered under the coffee table',
                'single minimalist floor lamp beside the sofa',
            ],
            'alt_pool': [
                'one abstract framed wall art piece',
                'small potted snake plant in a concrete planter',
                'a single accent armchair angled toward the sofa',
            ],
        },
        'bedroom': {
            'items': [
                'low platform bed with a grey upholstered headboard against the back wall',
                'two matching minimalist nightstands with slim brass lamps',
                'a simple wood dresser against the side wall',
                'a neutral wool area rug under the bed',
            ],
            'alt_pool': [
                'a floor-length mirror in the corner',
                'sheer white curtains framing the window',
                'a small reading chair by the window',
            ],
        },
        'home_office': {
            'items': [
                'a white minimalist desk against the wall facing the window',
                'an ergonomic black mesh office chair',
                'a wall-mounted monitor arm with a single monitor',
                'an open wood shelving unit for books',
            ],
            'alt_pool': [
                'a small potted plant on the desk corner',
                'a slim floor lamp beside the desk',
                'a neutral rug under the desk area',
            ],
        },
    },
    'scandinavian': {
        'living_room': {
            'items': [
                'light-oak 3-seat sofa with linen cushions against the right wall',
                'round light-wood coffee table in front of the sofa',
                'slim walnut TV console with a mounted TV on the wall opposite the window',
                'jute area rug under the coffee table',
                'black arc floor lamp beside the sofa',
            ],
            'alt_pool': [
                'one large potted fig plant in the far corner',
                'a woven wall hanging above the sofa',
                'a pair of light-wood accent chairs',
            ],
        },
        'bedroom': {
            'items': [
                'light-oak bed frame with white linen bedding against the back wall',
                'two light-wood nightstands with woven-shade lamps',
                'a cream wool area rug under the bed',
                'a simple light-wood wardrobe against the side wall',
            ],
            'alt_pool': [
                'a small potted plant on a nightstand',
                'linen curtains framing the window',
                'a woven pouf at the foot of the bed',
            ],
        },
        'home_office': {
            'items': [
                'a light-oak desk against the wall facing the window',
                'a light fabric-upholstered office chair',
                'open light-wood shelving with books and a plant',
                'a cream area rug under the desk',
            ],
            'alt_pool': [
                'a woven pendant light above the desk',
                'a small potted plant on the desk',
                'a linen-shade desk lamp',
            ],
        },
    },
    'industrial': {
        'living_room': {
            'items': [
                'brown leather sofa against the main wall',
                'reclaimed-wood and black-metal coffee table',
                'black metal-framed TV console with mounted TV opposite the seating',
                'a dark patterned area rug under the coffee table',
                'an exposed-bulb floor lamp beside the sofa',
            ],
            'alt_pool': [
                'a metal-framed wall shelf with books',
                'a large potted plant in a metal planter',
                'black metal wall art with an urban theme',
            ],
        },
        'bedroom': {
            'items': [
                'black metal-frame bed with dark linen bedding against the back wall',
                'reclaimed-wood nightstands with exposed-bulb lamps',
                'a dark area rug under the bed',
                'a black metal clothing rack against the side wall',
            ],
            'alt_pool': [
                'a vintage leather bench at the foot of the bed',
                'exposed-bulb string lights along the wall',
                'a small potted plant on a nightstand',
            ],
        },
        'home_office': {
            'items': [
                'a reclaimed-wood desk on black metal legs against the wall facing the window',
                'a black leather and metal office chair',
                'black metal pipe shelving with books',
                'a dark rug under the desk',
            ],
            'alt_pool': [
                'an exposed-bulb desk lamp',
                'a metal wall organizer above the desk',
                'a small potted plant on the desk',
            ],
        },
    },
    'bohemian': {
        'living_room': {
            'items': [
                'a rattan-framed sofa with colorful patterned cushions against the main wall',
                'a low carved-wood coffee table',
                'a woven-textile-draped TV console with mounted TV opposite the seating',
                'a layered patterned area rug under the coffee table',
                'a woven rattan floor lamp beside the sofa',
            ],
            'alt_pool': [
                'several potted plants clustered in one corner',
                'a macrame wall hanging',
                'floor cushions near the coffee table',
            ],
        },
        'bedroom': {
            'items': [
                'a rattan headboard bed with layered patterned textiles against the back wall',
                'mismatched carved-wood nightstands with patterned-shade lamps',
                'a patterned area rug layered under the bed',
                'a woven basket storage unit against the side wall',
            ],
            'alt_pool': [
                'a hanging rattan chair in the corner',
                'a macrame wall hanging above the bed',
                'a potted plant on a nightstand',
            ],
        },
        'home_office': {
            'items': [
                'a carved-wood desk against the wall facing the window',
                'a rattan-back office chair with a patterned cushion',
                'open rattan shelving with books and plants',
                'a patterned rug under the desk',
            ],
            'alt_pool': [
                'a woven pendant light above the desk',
                'a small cluster of potted plants on the desk',
                'a patterned throw on the chair',
            ],
        },
    },
    'luxury_contemporary': {
        'living_room': {
            'items': [
                'a tufted velvet sofa in a jewel tone against the main wall',
                'a marble and gold-accent coffee table',
                'a gold-framed TV console with mounted TV opposite the seating',
                'a plush high-pile area rug under the coffee table',
                'a sculptural gold floor lamp beside the sofa',
            ],
            'alt_pool': [
                'a large statement mirror on the side wall',
                'a crystal-accent pendant light',
                'a pair of velvet accent chairs',
            ],
        },
        'bedroom': {
            'items': [
                'an upholstered velvet bed with a tall tufted headboard against the back wall',
                'matching marble-top nightstands with crystal-accent lamps',
                'a plush area rug under the bed',
                'a mirrored dresser against the side wall',
            ],
            'alt_pool': [
                'a velvet bench at the foot of the bed',
                'floor-length drapery framing the window',
                'a chandelier-style ceiling light',
            ],
        },
        'home_office': {
            'items': [
                'a marble-top desk with gold legs against the wall facing the window',
                'a tufted velvet office chair',
                'gold-accent open shelving with books',
                'a plush rug under the desk',
            ],
            'alt_pool': [
                'a crystal-accent desk lamp',
                'a statement mirror on the side wall',
                'a small gold-accent plant stand',
            ],
        },
    },
}

STYLE_LABELS = {
    'modern_minimalist':    'modern minimalist',
    'scandinavian':         'Scandinavian',
    'industrial':           'industrial',
    'bohemian':             'bohemian',
    'luxury_contemporary':  'luxury contemporary',
}


def compile_edit_prompt(style, room_type, variation_index, rng):
    # Builds one FLUX.2 edit instruction for a given style/room/variation.
    # Varies the alt_pool selection per variation_index so the 3 outputs are
    # genuinely different layouts, not near-duplicates of the same composition.
    if style not in STYLE_TEMPLATES:
        raise ValueError(
            f'Unknown style {style!r}. '
            f'Valid styles: {list(STYLE_TEMPLATES.keys())}'
        )
    if room_type not in STYLE_TEMPLATES[style]:
        raise ValueError(
            f'Unknown room_type {room_type!r} for style {style!r}. '
            f'Valid room types: {list(STYLE_TEMPLATES[style].keys())}'
        )

    template   = STYLE_TEMPLATES[style][room_type]
    items      = list(template['items'])
    alt_pool   = list(template['alt_pool'])

    # Deterministic-but-varied alt-item selection per variation, seeded by variation_index
    # so re-running the same seed reproduces the same layout every time.
    rng.shuffle(alt_pool)
    n_alts = 2 if len(alt_pool) >= 2 else len(alt_pool)
    items += alt_pool[:n_alts]

    style_label = STYLE_LABELS[style]
    room_label  = room_type.replace('_', ' ')
    item_list   = ', '.join(items)

    prompt = (
        f'Add {style_label}-style furniture to this empty {room_label}: {item_list}. '
        f'Keep the walls, floor, ceiling, window, and camera perspective exactly unchanged. '
        f'Photorealistic lighting consistent with the existing light source in the photo. '
        f'Correct scale, correct perspective, natural shadows. '
        f'Do not add any furniture beyond what is listed above.'
    )
    return prompt


# Verify completeness: all 5 styles x all 3 room types must be populated
_missing = []
for _s in STYLES:
    for _r in ROOM_TYPES:
        if _s not in STYLE_TEMPLATES or _r not in STYLE_TEMPLATES[_s]:
            _missing.append(f'{_s}/{_r}')
        elif not STYLE_TEMPLATES[_s][_r]['items'] or not STYLE_TEMPLATES[_s][_r]['alt_pool']:
            _missing.append(f'{_s}/{_r} (empty lists)')
if _missing:
    raise AssertionError(f'STYLE_TEMPLATES incomplete: {_missing}')

# Test ValueError on bad input
try:
    compile_edit_prompt('bad_style', 'living_room', 0, random.Random(0))
    raise AssertionError('Expected ValueError for bad style')
except ValueError:
    pass

# Test a real prompt
_rng_test = random.Random(42)
_test_prompt = compile_edit_prompt('scandinavian', 'living_room', 0, _rng_test)
assert 'Scandinavian' in _test_prompt
print('✅ STYLE_TEMPLATES complete (5 styles x 3 room types).')
print('✅ compile_edit_prompt() defined and verified.')
print(f'   Sample prompt: {_test_prompt[:120]}...')


## Section 13 — Core Inference Function


In [ ]:
def generate_variations(image, style, room_type):
    # Core inference entrypoint.
    # Returns a dict matching the shape expected by the FastAPI /infer response.
    overall_start = time.time()

    prep_image = prepare_image(image)
    print(f'  Input prepared: {prep_image.size}')

    # Depth pass -- metadata/debugging only, not fed into FLUX.2 (see Section 11b).
    depth_start = time.time()
    depth_map   = get_depth_map(prep_image)
    depth_time  = time.time() - depth_start

    variations  = []
    render_start = time.time()
    for i, seed in enumerate(SEEDS[:NUM_VARIATIONS]):
        rng    = random.Random(seed)
        prompt = compile_edit_prompt(style, room_type, i, rng)
        print(f'  Variation {i+1}/3 (seed={seed}): {prompt[:80]}...')

        gen = torch.Generator(device='cuda').manual_seed(seed)
        out_image = pipe(
            image=prep_image,
            prompt=prompt,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=gen,
        ).images[0]

        variations.append({'seed': seed, 'prompt': prompt, 'image': out_image})

    render_time = time.time() - render_start

    # Sanity check: outputs must not be identical (guards against seed/prompt being ignored).
    hashes = [hash(v['image'].tobytes()) for v in variations]
    if len(set(hashes)) < len(hashes):
        print(
            '⚠️  WARNING: two or more variations produced identical output images. '
            'Check that SEEDS are distinct and compile_edit_prompt alt-pool shuffle is '
            'seeded per-variation.'
        )

    total_time = time.time() - overall_start

    import diffusers as _dmod
    return {
        'style':              style,
        'room_type':          room_type,
        'variations':         variations,
        'depth_map':          depth_map,
        'depth_time_sec':     round(depth_time, 2),
        'render_time_sec':    round(render_time, 2),
        'total_time_sec':     round(total_time, 2),
        'model_id':           MODEL_ID,
        'diffusers_version':  _dmod.__version__,
    }


def refine_variation(image, style, room_type, base_prompt, seed):
    # Optional single low-strength refine pass for shadow/lighting coherence.
    # Only called on-demand, not for all 3 variations.
    #
    # Note on FLUX.2 klein 'strength': the pipeline may not accept a numeric
    # strength= kwarg the way img2img SDXL does. The approach here uses a
    # conservative edit instruction (not a lower step count with strength=)
    # which is the safest cross-version approach for this pipeline class.
    # If the installed build later adds strength= support, consider switching.
    gen = torch.Generator(device='cuda').manual_seed(seed + 1)
    refine_prompt = (
        f'{base_prompt} Subtly harmonize lighting, shadows, and color grading between '
        f'the added furniture and the original room photo. '
        f'Do not change furniture placement or style.'
    )
    result = pipe(
        image=image,
        prompt=refine_prompt,
        num_inference_steps=max(4, int(NUM_INFERENCE_STEPS * REFINE_STRENGTH)),
        guidance_scale=GUIDANCE_SCALE,
        generator=gen,
    ).images[0]
    return result


print('✅ generate_variations() defined (FLUX.2 edit mode, 3 variations, hash-checked).')
print('✅ refine_variation() defined (conservative instruction-based refine pass).')


## Section 14 — Metadata & Response Assembly


In [ ]:
def pil_to_b64(img):
    # Encode PIL image to base64 PNG string.
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('utf-8')


print('✅ pil_to_b64() defined.')


## Section 15 — Sanity Check & Visualization

Runs a full end-to-end inference call on a placeholder room image.  
**This cell fails loudly** (assert) if:
- Any variation image is `None`
- Any two variations are pixel-identical (hash check)
- Generation took unreasonably long (>120s)

The 3 variation outputs are displayed side by side. MLSD/depth maps are no longer plotted (those models are removed).


In [ ]:
import hashlib

# Create a placeholder room image: empty white-walled room with a wooden floor
# (neutral enough to trigger meaningful edits from the model)
_sanity_arr = np.ones((512, 512, 3), dtype=np.uint8) * 245  # off-white walls
_sanity_arr[350:, :, 0] = 200  # warm-wood floor tone
_sanity_arr[350:, :, 1] = 160
_sanity_arr[350:, :, 2] = 120
_sanity_arr[30:320, 180:330, :] = 220  # window lighter area
_sanity_pil = Image.fromarray(_sanity_arr)

print('Running sanity check (modern_minimalist + living_room)...')
_t_sanity = time.time()

try:
    _sanity_result = generate_variations(_sanity_pil, 'modern_minimalist', 'living_room')
    _elapsed = time.time() - _t_sanity

    # Assert 1: all 3 variations returned and are non-None
    assert len(_sanity_result['variations']) == NUM_VARIATIONS, (
        f'Expected {NUM_VARIATIONS} variations, got {len(_sanity_result["variations"])}'
    )
    for _v in _sanity_result['variations']:
        assert _v['image'] is not None, f'Variation seed={_v["seed"]} returned None image'

    # Assert 2: no two variations are pixel-identical
    _hashes = [
        hashlib.md5(_v['image'].tobytes()).hexdigest()
        for _v in _sanity_result['variations']
    ]
    assert len(set(_hashes)) == len(_hashes), (
        f'Two or more variations are pixel-identical. Hashes: {_hashes}'
    )

    # Assert 3: timing within budget
    assert _elapsed < 120, f'Sanity check took {_elapsed:.1f}s (>120s budget)'

    # Plot
    _fig, _axes = plt.subplots(1, NUM_VARIATIONS, figsize=(6 * NUM_VARIATIONS, 5))
    _fig.suptitle(
        f'Sanity Check -- modern_minimalist / living_room | {_elapsed:.1f}s | '
        f'FLUX.2 klein (diffusers {_sanity_result["diffusers_version"]})',
        fontsize=11
    )
    for _ax, _v in zip(_axes, _sanity_result['variations']):
        _ax.imshow(_v['image'])
        _ax.set_title(f'seed={_v["seed"]}')
        _ax.axis('off')
    plt.tight_layout()
    plt.savefig('/content/sanity_check_output.png', dpi=100, bbox_inches='tight')
    plt.show()

    print(f'✅ Sanity check passed in {_elapsed:.1f}s.')
    print(f'   render_time={_sanity_result["render_time_sec"]}s  depth_time={_sanity_result["depth_time_sec"]}s')

except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    print('❌ OOM during sanity check. Runtime -> Factory reset runtime -> re-run from Section 2.')
    raise
except AssertionError as _ae:
    print(f'❌ Sanity assertion failed: {_ae}')
    raise
except Exception as _e:
    print(f'❌ Sanity check failed: {_e}')
    raise


## Section 16 — FastAPI App Definition


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger('roomcanvas')

app = FastAPI(
    title='RoomCanvas Inference Server',
    description=(
        'FLUX.2 [klein] 4B instruction-based room redesign. '
        'POST /infer with image + style + room_type to get 3 redesigned variations.'
    ),
    version='3.0.0',
)

print('✅ FastAPI app defined.')


## Section 17 — Health Check Endpoint


In [ ]:
@app.get('/health')
def health():
    # Return server status and GPU stats.
    # Polled by the startup cell -- MUST return HTTP 200 before ngrok is created.
    import diffusers as _dmod
    return {
        'status':              'ok',
        'model_id':            MODEL_ID,
        'depth_model_id':      DEPTH_MODEL_ID,
        'diffusers_version':   _dmod.__version__,
        'gpu': (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
        ),
        'vram_allocated_mb': (
            round(torch.cuda.memory_allocated() / 1024**2, 1)
            if torch.cuda.is_available() else 0
        ),
        'vram_total_mb': (
            round(torch.cuda.get_device_properties(0).total_memory / 1024**2, 1)
            if torch.cuda.is_available() else 0
        ),
    }


print('✅ GET /health defined.')


## Section 18 — Main /infer Endpoint

> ⚠️ **API contract change:** `room_type` is now a **required** form field (previously auto-classified by CLIP).
> The frontend must include a room-type selector (`living_room` / `bedroom` / `home_office`) or requests will return 400.


In [ ]:
@app.post('/infer')
async def infer(
    image:     UploadFile = File(...),
    style:     str        = Form(...),
    room_type: str        = Form(...),
):
    # Validate inputs
    if style not in STYLE_TEMPLATES:
        raise HTTPException(
            status_code=400,
            detail=f'Invalid style {style!r}. Valid: {list(STYLE_TEMPLATES.keys())}'
        )
    if room_type not in ROOM_TYPES:
        raise HTTPException(
            status_code=400,
            detail=f'Invalid room_type {room_type!r}. Valid: {ROOM_TYPES}'
        )

    # Load and validate image
    try:
        contents = await image.read()
        pil_image = Image.open(io.BytesIO(contents))
    except Exception:
        raise HTTPException(status_code=400, detail='Uploaded file is not a valid image.')

    # Run inference
    try:
        result = generate_variations(pil_image, style, room_type)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise HTTPException(
            status_code=503,
            detail='GPU ran out of memory. Try again in a moment, or reduce image resolution.'
        )
    except ValueError as ve:
        # compile_edit_prompt raises ValueError on bad style/room_type combos
        raise HTTPException(status_code=400, detail=str(ve))
    except Exception as e:
        logger.error(f'/infer failed: {e}')
        raise HTTPException(status_code=500, detail=f'Generation failed: {e}')

    # Build response
    response_variations = []
    for v in result['variations']:
        response_variations.append({
            'seed':      v['seed'],
            'prompt':    v['prompt'],
            'image_b64': pil_to_b64(v['image']),
        })

    return JSONResponse({
        'style':             result['style'],
        'room_type':         result['room_type'],
        'variations':        response_variations,
        'depth_time_sec':    result['depth_time_sec'],
        'render_time_sec':   result['render_time_sec'],
        'total_time_sec':    result['total_time_sec'],
        'model_id':          result['model_id'],
        'diffusers_version': result['diffusers_version'],
    })


print('✅ POST /infer defined (style + room_type required, 400/503/500 on failures).')


## Section 19 — Server Startup (Background)

> Starts uvicorn in a daemon thread using the Python 3.12-safe `uvicorn.Server` API,
> then polls `/health` until the server accepts connections.


In [ ]:
nest_asyncio.apply()

# Python 3.12 no longer auto-creates an event loop in worker threads.
# Fix: use uvicorn.Server API + create a fresh event loop inside the thread.
config = uvicorn.Config(
    app,
    host='0.0.0.0',
    port=8000,
    loop='asyncio',
    reload=False,
    workers=1,
    access_log=False,
    log_level='warning',
)
server = uvicorn.Server(config)


def run_server():
    # Thread target: creates its own event loop (required for Python 3.12).
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(server.serve())
    finally:
        loop.close()


print('🚀 Starting uvicorn (Python 3.12-safe background thread)...')
server_thread = threading.Thread(target=run_server, daemon=True, name='uvicorn-server')
server_thread.start()

print('⏳ Waiting for http://127.0.0.1:8000/health ...')
server_up = False
for attempt in range(30):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            server_up = True
            break
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        pass
    time.sleep(1)

if server_up:
    print('✅ Server reachable on http://127.0.0.1:8000')
    print('   Run Section 20 to open the ngrok tunnel.')
else:
    print('❌ Server did NOT come up in 30s. Thread alive:', server_thread.is_alive())
    raise RuntimeError(
        'FastAPI/uvicorn failed to start on port 8000. '
        'Check that no other process owns port 8000 and re-run this cell.'
    )


## Section 20 — Ngrok Tunnel Setup

**Get your free ngrok auth token:** sign up at [ngrok.com](https://ngrok.com/), paste the token into Section 5.  
The public URL **changes every restart** — paste the new URL into `backend/.env` as `COLAB_INFERENCE_URL`.

If `NGROK_AUTH_TOKEN` is empty, the cell below falls back to a **Cloudflare quick tunnel** (free, more stable URL for demo day).


In [ ]:
public_url = None

if NGROK_AUTH_TOKEN:
    print('🌐 Creating ngrok tunnel...')
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        tunnel     = ngrok.connect('127.0.0.1:8000')
        public_url = tunnel.public_url
        print(f'✅ ngrok tunnel: {public_url}')
    except Exception as _ngrok_err:
        print(f'⚠️  ngrok failed: {_ngrok_err}. Trying Cloudflare fallback...')

if public_url is None:
    # Cloudflare quick tunnel -- free, no auth token required.
    print('🌐 Attempting Cloudflare quick tunnel fallback...')
    import subprocess as _sp, re as _re, threading as _thr
    _cf_output = []
    def _run_cf():
        _proc = _sp.Popen(
            ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
            stderr=_sp.PIPE, stdout=_sp.DEVNULL, text=True
        )
        for _line in _proc.stderr:
            _cf_output.append(_line)
            if 'trycloudflare.com' in _line:
                break
    _cf_thread = _thr.Thread(target=_run_cf, daemon=True)
    _cf_thread.start()
    _cf_thread.join(timeout=15)
    for _line in _cf_output:
        _m = _re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', _line)
        if _m:
            public_url = _m.group(0)
            print(f'✅ Cloudflare tunnel: {public_url}')
            break

if public_url is None:
    raise RuntimeError(
        'Both ngrok and Cloudflare tunnel failed. '
        'Paste a valid NGROK_AUTH_TOKEN in Section 5 and re-run.'
    )

print(f'\n✅ Public endpoint ready!')
print(f'   Health: {public_url}/health')
print(f'   Infer:  POST {public_url}/infer  (fields: image, style, room_type)')
print(f'\n📋 Paste into backend/.env: COLAB_INFERENCE_URL={public_url}')


## Section 21 — Live Test Cell

**Run after Section 20.** Tests all 5 valid styles × 1 room type, one invalid style (expect 400), one invalid room_type (expect 400), and a corrupt upload (expect 400).  
Asserts each response contains exactly 3 distinct, valid base64 images.


In [ ]:
from google.colab import files as colab_files

test_img_path = '/content/test_room.jpg'
if not os.path.exists(test_img_path):
    print('Upload a room photo (JPG/PNG):')
    _uploaded = colab_files.upload()
    if _uploaded:
        test_img_path = list(_uploaded.keys())[0]
    else:
        print('No file uploaded -- using neutral grey stand-in.')
        Image.fromarray(np.full((512, 512, 3), 180, dtype=np.uint8)).save(test_img_path)

STYLES_TO_TEST = STYLES  # all 5
TEST_ROOM_TYPE = 'living_room'
SERVER_URL     = str(public_url)

test_results = []
print(f'Testing {len(STYLES_TO_TEST)} styles + 2 invalid requests against {SERVER_URL}/infer...\n')

for style_name in STYLES_TO_TEST:
    print(f'  -> {style_name} / {TEST_ROOM_TYPE} ... ', end='', flush=True)
    try:
        with open(test_img_path, 'rb') as fh:
            resp = requests.post(
                f'{SERVER_URL}/infer',
                files={'image': ('room.jpg', fh, 'image/jpeg')},
                data={'style': style_name, 'room_type': TEST_ROOM_TYPE},
                timeout=120,
            )
        if resp.status_code == 200:
            data = resp.json()
            # Assert response shape
            assert len(data['variations']) == 3, 'Expected 3 variations'
            # Assert all images are valid non-empty base64
            for _v in data['variations']:
                _decoded = base64.b64decode(_v['image_b64'])
                assert len(_decoded) > 1000, f'Image suspiciously small: {len(_decoded)} bytes'
            # Assert all 3 images are distinct
            import hashlib
            _hashes = [hashlib.md5(base64.b64decode(_v['image_b64'])).hexdigest() for _v in data['variations']]
            assert len(set(_hashes)) == 3, f'Duplicate variation images: {_hashes}'
            test_results.append((style_name, data))
            print(
                f'OK  total={data["total_time_sec"]}s '
                f'render={data["render_time_sec"]}s '
                f'depth={data["depth_time_sec"]}s'
            )
        else:
            print(f'FAIL HTTP {resp.status_code}: {resp.text[:200]}')
    except Exception as e:
        print(f'ERROR: {e}')

# Display outputs
for style_name, data in test_results:
    _fig, _axes = plt.subplots(1, 3, figsize=(15, 5))
    _fig.suptitle(
        f'{style_name} | {data["room_type"]} | {data["total_time_sec"]}s',
        fontsize=11
    )
    for _ax, _v in zip(_axes, data['variations']):
        _pil_v = Image.open(io.BytesIO(base64.b64decode(_v['image_b64'])))
        _ax.imshow(_pil_v); _ax.set_title(f'Seed {_v["seed"]}'); _ax.axis('off')
    plt.tight_layout(); plt.show()

# Error case tests
print('\nTesting invalid style key (expect HTTP 400)...')
with open(test_img_path, 'rb') as fh:
    _bad = requests.post(
        f'{SERVER_URL}/infer',
        files={'image': ('r.jpg', fh, 'image/jpeg')},
        data={'style': 'not_valid', 'room_type': 'living_room'},
        timeout=10,
    )
assert _bad.status_code == 400, f'Expected 400, got {_bad.status_code}'
print(f'OK HTTP 400: {_bad.json()["detail"][:80]}')

print('Testing invalid room_type (expect HTTP 400)...')
with open(test_img_path, 'rb') as fh:
    _bad2 = requests.post(
        f'{SERVER_URL}/infer',
        files={'image': ('r.jpg', fh, 'image/jpeg')},
        data={'style': 'scandinavian', 'room_type': 'bathroom'},
        timeout=10,
    )
assert _bad2.status_code == 400, f'Expected 400, got {_bad2.status_code}'
print(f'OK HTTP 400: {_bad2.json()["detail"][:80]}')

print('Testing corrupt file upload (expect HTTP 400)...')
_bad3 = requests.post(
    f'{SERVER_URL}/infer',
    files={'image': ('bad.jpg', b'not an image', 'image/jpeg')},
    data={'style': 'scandinavian', 'room_type': 'living_room'},
    timeout=10,
)
assert _bad3.status_code == 400, f'Expected 400, got {_bad3.status_code}'
print(f'OK HTTP 400: {_bad3.json()["detail"][:80]}')

print(f'\n✅ Live test complete. {len(test_results)}/{len(STYLES_TO_TEST)} styles passed.')


## Section 22 — Performance Diagnostics


In [ ]:
vram_alloc = torch.cuda.memory_allocated() / 1024**3
vram_res   = torch.cuda.memory_reserved()  / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print('-' * 52)
print('  VRAM Diagnostics')
print('-' * 52)
print(f'  Allocated : {vram_alloc:.2f} GB')
print(f'  Reserved  : {vram_res:.2f} GB')
print(f'  Total     : {vram_total:.1f} GB')
print(f'  Free      : {vram_total - vram_res:.2f} GB')
print(f'  Peak budget per run: ~9-14 GB (FLUX.2 klein + Depth Anything + CPU offload)')

if test_results:
    total_times  = [d['total_time_sec']  for _, d in test_results]
    render_times = [d['render_time_sec'] for _, d in test_results]
    depth_times  = [d['depth_time_sec']  for _, d in test_results]
    avg_total    = sum(total_times) / len(total_times)

    print()
    print('-' * 52)
    print('  Timing Breakdown (from live test)')
    print('-' * 52)
    for sn, d in test_results:
        print(
            f'  {sn:25s}  total={d["total_time_sec"]:5.1f}s  '
            f'render={d["render_time_sec"]:5.1f}s  depth={d["depth_time_sec"]:4.1f}s'
        )
    print(f'  {"AVERAGE":25s}  total={avg_total:5.1f}s')

    THRESHOLD = 40
    print()
    if avg_total <= THRESHOLD:
        print(f'  ✅ Average total: {avg_total:.1f}s -- within budget (<{THRESHOLD}s)')
    else:
        print(f'  ⚠️  Average total: {avg_total:.1f}s -- EXCEEDS {THRESHOLD}s.')
        print('      Consider reducing IMG_MAX_SIDE in Section 5.')
else:
    print('\n  ⚠️  No test_results -- run Section 21 first.')


## Section 23 — Outputs, Export & Download

Saves all generated images, pipeline metadata, and inference metrics to `outputs/`, then zips for download.  
**Run after Section 21 has completed successfully.**


In [ ]:
import zipfile, json as _json_mod
from datetime import datetime

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

DIRS = ['/content/outputs/images', '/content/outputs/reports', '/content/outputs/model']
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print('✅ Output directories created.')

saved_images = []
if 'test_results' in dir() and test_results:
    for style_name, data in test_results:
        style_dir = f'/content/outputs/images/{style_name}'
        os.makedirs(style_dir, exist_ok=True)
        for v in data['variations']:
            img   = Image.open(io.BytesIO(base64.b64decode(v['image_b64'])))
            fname = f'{style_dir}/seed_{v["seed"]}.png'
            img.save(fname, dpi=(300, 300))
            saved_images.append(fname)
    print(f'✅ Saved {len(saved_images)} images to outputs/images/')
else:
    print('⚠️  test_results not found -- run Section 21 first.')

import diffusers as _dmod
model_info_lines = [
    'RoomCanvas AI -- Model & Pipeline Information',
    f'Generated: {TIMESTAMP}',
    '',
    '=== Inference Pipeline ===',
    f'Primary model:   {MODEL_ID}  (FLUX.2 klein 4B, edit mode)',
    f'Depth model:     {DEPTH_MODEL_ID}  (metadata/debugging only)',
    f'diffusers:       {_dmod.__version__}',
    '',
    '=== Generation Parameters ===',
    f'Inference steps: {NUM_INFERENCE_STEPS}',
    f'Guidance scale:  {GUIDANCE_SCALE}',
    f'Max image side:  {IMG_MAX_SIDE}px (multiple of {IMG_DIM_MULTIPLE})',
    f'Seeds:           {SEEDS}',
    f'Variations:      {NUM_VARIATIONS}',
    '',
    '=== Styles Supported ===',
    ', '.join(STYLES),
    '',
    '=== Room Types Supported ===',
    ', '.join(ROOM_TYPES),
]
with open('/content/outputs/model/model_info.txt', 'w') as fh:
    fh.write('\n'.join(model_info_lines))
print('✅ Saved outputs/model/model_info.txt')

metrics = {
    'timestamp':            TIMESTAMP,
    'model_id':             MODEL_ID,
    'depth_model_id':       DEPTH_MODEL_ID,
    'diffusers_version':    _dmod.__version__,
    'num_inference_steps':  NUM_INFERENCE_STEPS,
    'guidance_scale':       GUIDANCE_SCALE,
    'img_max_side':         IMG_MAX_SIDE,
    'seeds':                SEEDS,
    'num_variations':       NUM_VARIATIONS,
}
if 'test_results' in dir() and test_results:
    tts   = [d['total_time_sec']  for _, d in test_results]
    rts   = [d['render_time_sec'] for _, d in test_results]
    dts   = [d['depth_time_sec']  for _, d in test_results]
    metrics.update({
        'avg_total_time_sec':  round(sum(tts) / len(tts), 2),
        'avg_render_time_sec': round(sum(rts) / len(rts), 2),
        'avg_depth_time_sec':  round(sum(dts) / len(dts), 2),
        'styles_tested':       [s for s, _ in test_results],
    })
with open('/content/outputs/reports/inference_metrics.json', 'w') as fh:
    _json_mod.dump(metrics, fh, indent=2)
print('✅ Saved outputs/reports/inference_metrics.json')

zip_path = f'/content/RoomCanvas_Output_{TIMESTAMP}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk('/content/outputs'):
        for fname in fnames:
            full    = os.path.join(root, fname)
            arcname = os.path.relpath(full, '/content')
            zf.write(full, arcname)
print(f'✅ Created {zip_path}')

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
    print('✅ Download triggered.')
except Exception as e:
    print(f'⚠️  Auto-download failed ({e}). Download manually from: {zip_path}')


## Section 24 — Cleanup (Optional)

Run only when done. Restart the runtime before starting a new session.


In [ ]:
try:
    ngrok.disconnect(public_url)
    print('✅ ngrok tunnel disconnected.')
except Exception as e:
    print(f'⚠️  ngrok disconnect: {e}')

try:
    del pipe, depth_estimator
    torch.cuda.empty_cache()
    print('✅ Models deleted and CUDA cache cleared.')
except Exception as e:
    print(f'⚠️  Cleanup: {e}')

print('\nResources released. Restart runtime before next session for a clean state.')


## Section 25 — Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `AttributeError`/`ImportError` on `Flux2KleinPipeline` | diffusers not installed from `main` | Re-run Section 3 |
| `TypeError: from_pretrained() got an unexpected keyword argument 'dtype'` (or `torch_dtype`) | Defensive fallback in Section 7 should catch this; if still failing, the kwarg changed again | Print `diffusers.__version__` and check the current model card |
| `CUDA OOM` during `/infer` | `enable_model_cpu_offload()` should be active from Section 7; still OOMing means the image is too large | Reduce `IMG_MAX_SIDE` in Section 5 (try 768) |
| `ValueError: Unknown room_type` | Frontend/test client sending a value not in `ROOM_TYPES` | Check Section 5's `ROOM_TYPES` list matches the frontend dropdown |
| Depth pass warning in logs but generation completes | Expected — depth is non-fatal metadata-only (Section 11b) | Does not block a response; safe to ignore |
| Two variations look identical | Hash-warning in `generate_variations` fired | Check that `SEEDS` are distinct and `compile_edit_prompt` alt-pool shuffle is seeded per-variation |
| Section 7 edit-mode validation fails | `pipe(image=...)` API not supported on installed diffusers build | Re-run Section 3 to install from `main`; check printed `diffusers.__version__` |
| ngrok URL not working | Session restarted, or free-tier tunnel expired | Re-run Section 20, or use the Cloudflare fallback (runs automatically if NGROK_AUTH_TOKEN is empty) |
| Server not up after 30s | Port conflict or uvicorn crash | Check for another process on port 8000, re-run Section 19 |
| `/infer` returns 400 on valid style | Style key typo | Valid: `modern_minimalist`, `scandinavian`, `industrial`, `bohemian`, `luxury_contemporary` |
| Generation time >40s | Model doing many steps | Check `NUM_INFERENCE_STEPS` in Section 5; for distilled klein try 4–8 steps |
| `No GPU detected` | Wrong runtime type | Runtime → Change runtime type → T4 GPU → Save |
